In [1]:
import subprocess
import threading

The working directory contains the PEST++ input files (template directory) and the script to run the workers (worker.py; there's another script called worker_fs.py but we'll get tp that later). This script is what we run to start the master process and setup the workers to perform patrallel runs of PEST++ tools.

First, let's try running worker.py to see if it works (SPOILER: IT DOESN'T! Stop the process after about 30 seconds, so you don't keep waiting forever!). Let's print all the output and error messages so we can see what's going on.

In [2]:
def read_output(process):
    for line in iter(process.stdout.readline, ''):
        print(line, end='')

def read_error(process):
    for line in iter(process.stderr.readline, ''):
        print(line, end='')

In [3]:
command = ["python", "worker.py"]
process = subprocess.Popen(command, cwd=".", stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

stdout_thread = threading.Thread(target=read_output, args=(process,))
stderr_thread = threading.Thread(target=read_error, args=(process,))
stdout_thread.start()
stderr_thread.start()

process.wait()

stdout_thread.join()
stderr_thread.join()

master:pestpp-mou pest.pst /h :4004 in hosaki_model_master


             pestpp-mou: multi-objective optimization under uncertainty

                   by the PEST++ development team


version: 5.2.17
binary compiled on Jan  6 2025 at 09:08:29

started at 01/08/25 10:44:40
...processing command line: ' pestpp-mou pest.pst /h :4004'
...using panther run manager in master mode using port 4004

using control file: "pest.pst"
in directory: "c:\Users\rmacasieb\Documents\GitHub\pyemu-rqm\examples\freeze_support\hosaki_model_master"
processing control file pest.pst
Note: 3 unused lines in pest control file, see rec file...


:~-._                                                 _.-~:
: :.~^o._        ________---------________        _.o^~.:.:
 : ::.`?88booo~~~.::::::::...::::::::::::..~~oood88P'.::.:
 :  ::: `?88P .:::....         ........:::::. ?88P' :::. :
  :  :::. `? .::.            . ...........:::. P' .:::. :
   :  :::   ... ..  ...       .. .::::......::.   :::. :
   `  :' .... ..  .:

Did you see the Permission Errors raised? This happens when multiple masters are started almost at the same time by accident which can happen when PyEmu uses the multiprocessing module to perform parallel runs. When multiple masters are started at the same time, they try to create the same directory at the same time, and this causes the Permission Error. To resolve this, we have to use freeze_support() prior to starting the workers.

During the Permission Error, another error may also occur and this is more particular to Windows users. This is because the multiprocessing module works differently in Windows compared to Mac or other Linux-based systems. To make things simple, we have to wrap start_workers in the if __name__ == '__main__': block.

Compare the two scripts below: the first one is worker.py, which is what we run previously and the second one is worker_fs.py, which is the modified version of worker.py that includes the freeze_support() function and the if __name__ == '__main__': block.


In [1]:
with open('worker.py', 'r') as file:
    contents = file.read()
    print(contents)

import sys
import pyemu

t_d = "hosaki_template"
m_d = "hosaki_model_master"
num_workers = 10
sys.path.insert(0,t_d)

from forward_run import hosaki_ppw_worker as ppw_function
pyemu.os_utils.start_workers(t_d,"pestpp-mou","pest.pst",
                                num_workers=num_workers,
                                master_dir=m_d,worker_root='.',
                                verbose=True,
                                ppw_function=ppw_function)



In [2]:
with open('worker_fs.py', 'r') as file:
    contents = file.read()
    print(contents)

import sys
import pyemu
import multiprocessing as mp

if __name__ == "__main__":
    mp.freeze_support()

    t_d = "hosaki_template"
    m_d = "hosaki_model_master_2"
    num_workers = 10
    sys.path.insert(0,t_d)

    from forward_run import hosaki_ppw_worker as ppw_function
    pyemu.os_utils.start_workers(t_d,"pestpp-mou","pest.pst",
                                    num_workers=num_workers,
                                    master_dir=m_d,worker_root='.',
                                    verbose=True,
                                    ppw_function=ppw_function)



Take note of the new lines added to the script. Mac and Linux-based users may not need these additional lines of code but it doesn't hurt to put them in as well.

Now let's try running the modified script.

In [4]:
import subprocess
import threading

def read_output(process):
    for line in iter(process.stdout.readline, ''):
        print(line, end='')

def read_error(process):
    for line in iter(process.stderr.readline, ''):
        print(line, end='')

command = ["python", "worker_fs.py"]
process = subprocess.Popen(command, cwd=".", stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

stdout_thread = threading.Thread(target=read_output, args=(process,))
stderr_thread = threading.Thread(target=read_error, args=(process,))
stdout_thread.start()
stderr_thread.start()

process.wait()

stdout_thread.join()
stderr_thread.join()

master:pestpp-mou pest.pst /h :4004 in hosaki_model_master_2


             pestpp-mou: multi-objective optimization under uncertainty

                   by the PEST++ development team


version: 5.2.17
binary compiled on Jan  6 2025 at 09:08:29

started at 01/08/25 10:48:47
...processing command line: ' pestpp-mou pest.pst /h :4004'
...using panther run manager in master mode using port 4004

using control file: "pest.pst"
in directory: "c:\Users\rmacasieb\Documents\GitHub\pyemu-rqm\examples\freeze_support\hosaki_model_master_2"
processing control file pest.pst
Note: 3 unused lines in pest control file, see rec file...


:~-._                                                 _.-~:
: :.~^o._        ________---------________        _.o^~.:.:
 : ::.`?88booo~~~.::::::::...::::::::::::..~~oood88P'.::.:
 :  ::: `?88P .:::....         ........:::::. ?88P' :::. :
  :  :::. `? .::.            . ...........:::. P' .:::. :
   :  :::   ... ..  ...       .. .::::......::.   :::. :
   `  :' .... ..

It should finish successfully now. Remember these two things when setting up and running pyworker in Windows (Mac and Linux users may need these as well):

- Run start_workers inside the if __name__ == '__main__': block
- Use freeze_support() prior to starting the workers
